## Why does character level tokenization fail?
This section compares word and character tokenization to show why character-level splitting can be inefficient.

In [ ]:
# Compare word-level tokens vs character-level tokens on the same sentence.
sentence = "Today, I want to start my day with a cup of coffee"

# Split on whitespace to get individual words.
words = sentence.split()

# Print the list of word tokens.
print("Word tokens:", words)

# Print the total number of words.
print("Number of words:", len(words))

# Reuse the sentence and split into individual characters.
sentence = "Today, I want to start my day with a cup of coffee"
characters = list(sentence)

# Print the list of character tokens and total count.
print("Character tokens:", characters)
print("Number of characters:", len(characters))

Word tokens: ['Today,', 'I', 'want', 'to', 'start', 'my', 'day', 'with', 'a', 'cup', 'of', 'coffee']
Number of words: 12
Character tokens: ['T', 'o', 'd', 'a', 'y', ',', ' ', 'I', ' ', 'w', 'a', 'n', 't', ' ', 't', 'o', ' ', 's', 't', 'a', 'r', 't', ' ', 'm', 'y', ' ', 'd', 'a', 'y', ' ', 'w', 'i', 't', 'h', ' ', 'a', ' ', 'c', 'u', 'p', ' ', 'o', 'f', ' ', 'c', 'o', 'f', 'f', 'e', 'e']
Number of characters: 50


## Implementing Byte Pair Encoding (BPE) from scratch
We will build BPE step by step and observe how repeated pairs are merged into new tokens.

### Step 1: Take raw text and tokenize into characters
Convert the corpus into integer token IDs so we can count and merge frequent adjacent pairs.

In [ ]:
# Create a training corpus and convert it into integer token IDs (ASCII-based).
text = """The Dark Knight Rises is a superhero movie released in 2012. It is the final part of Christopher Nolan’s Dark Knight trilogy, following Batman Begins and The Dark Knight. The film stars Christian Bale as Bruce Wayne/Batman, who has been retired as Batman for eight years after the events of the previous movie.

The main villain in the movie is Bane, played by Tom Hardy. Bane is a powerful and intelligent terrorist who threatens Gotham City with destruction. He forces Bruce Wayne to come out of retirement and become Batman again. Anne Hathaway plays Selina Kyle, also known as Catwoman, a skilled thief with her own agenda.

The movie is about Bruce Wayne’s struggle to overcome his physical and emotional challenges to save Gotham. It also shows themes of hope, sacrifice, and resilience. The film has many exciting action scenes, such as a plane hijack and a massive battle in Gotham.

In the end, Batman saves the city and inspires the people of Gotham. However, he is believed to have sacrificed his life. The movie ends with a twist, suggesting that Bruce Wayne is alive and has moved on to live a quiet life.

The Dark Knight Rises was a big success and is loved by many fans for its epic story, strong characters, and thrilling action.

Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother Jonathan. It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, and Michael Caine. Set in a dystopian future where Earth is suffering from catastrophic blight and famine, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

The screenplay had its origins in a script Jonathan developed in 2007 and was originally set to be directed by Steven Spielberg. Theoretical physicist Kip Thorne was an executive producer and scientific consultant on the film, and wrote the tie-in book The Science of Interstellar. It was Lynda Obst's final film as producer before her death. Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm. Filming began in late 2013 and took place in Alberta, Klaustur, and Los Angeles. Interstellar uses extensive practical and miniature effects, and the company DNEG created additional digital effects.

Interstellar was released in theaters on November 7, 2014. In the United States, it was first released on film stock, expanding to venues using digital projectors. The film received generally positive reviews and grossed $681 million worldwide during its initial theatrical run, making it the tenth-highest-grossing film of 2014. Among its various accolades, Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects.

"""

# Alternative (recommended) for full Unicode support: byte-level tokenization.
# tokens = text.encode("utf-8")
# tokens = list(map(int, tokens))

# Simpler demo path: map each character to its ASCII/Unicode code point.
tokens = [ord(ch) for ch in text]

In [ ]:
# Work on a copy so the original token sequence remains unchanged.
ids = list(tokens)

In [ ]:
# Inspect the initial token ID sequence before any merges.
print(ids)

[84, 104, 101, 32, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 32, 82, 105, 115, 101, 115, 32, 105, 115, 32, 97, 32, 115, 117, 112, 101, 114, 104, 101, 114, 111, 32, 109, 111, 118, 105, 101, 32, 114, 101, 108, 101, 97, 115, 101, 100, 32, 105, 110, 32, 50, 48, 49, 50, 46, 32, 73, 116, 32, 105, 115, 32, 116, 104, 101, 32, 102, 105, 110, 97, 108, 32, 112, 97, 114, 116, 32, 111, 102, 32, 67, 104, 114, 105, 115, 116, 111, 112, 104, 101, 114, 32, 78, 111, 108, 97, 110, 8217, 115, 32, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 32, 116, 114, 105, 108, 111, 103, 121, 44, 32, 102, 111, 108, 108, 111, 119, 105, 110, 103, 32, 66, 97, 116, 109, 97, 110, 32, 66, 101, 103, 105, 110, 115, 32, 97, 110, 100, 32, 84, 104, 101, 32, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 46, 32, 84, 104, 101, 32, 102, 105, 108, 109, 32, 115, 116, 97, 114, 115, 32, 67, 104, 114, 105, 115, 116, 105, 97, 110, 32, 66, 97, 108, 101, 32, 97, 115, 32, 66, 114, 117, 99, 101, 32, 87, 97, 121, 110, 101, 47

### Step 2: Write a function to count the frequency of adjacent token pairs
This frequency table tells us which pair should be merged next in BPE.

In [ ]:
# Count frequencies of adjacent token pairs in the current sequence.
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# Compute and print pair statistics for inspection.
stats = get_stats(ids)
print(stats)

{(84, 104): 13, (104, 101): 43, (101, 32): 76, (32, 68): 5, (68, 97): 4, (97, 114): 21, (114, 107): 4, (107, 32): 7, (32, 75): 7, (75, 110): 4, (110, 105): 8, (105, 103): 13, (103, 104): 9, (104, 116): 6, (116, 32): 30, (32, 82): 2, (82, 105): 2, (105, 115): 23, (115, 101): 9, (101, 115): 20, (115, 32): 64, (32, 105): 31, (32, 97): 56, (97, 32): 19, (32, 115): 26, (115, 117): 7, (117, 112): 2, (112, 101): 4, (101, 114): 28, (114, 104): 1, (114, 111): 16, (111, 32): 14, (32, 109): 17, (109, 111): 10, (111, 118): 10, (118, 105): 10, (105, 101): 16, (32, 114): 9, (114, 101): 23, (101, 108): 16, (108, 101): 13, (101, 97): 11, (97, 115): 22, (101, 100): 18, (100, 32): 40, (105, 110): 45, (110, 32): 41, (32, 50): 6, (50, 48): 6, (48, 49): 5, (49, 50): 1, (50, 46): 1, (46, 32): 19, (32, 73): 10, (73, 116): 4, (32, 116): 38, (116, 104): 42, (32, 102): 27, (102, 105): 18, (110, 97): 13, (97, 108): 19, (108, 32): 16, (32, 112): 15, (112, 97): 3, (114, 116): 3, (32, 111): 19, (111, 102): 9, (102,

### Step 3: Select the pair with the highest frequency
Choose the most common adjacent pair, since BPE merges the most frequent pair each iteration.

In [ ]:
# Pick the most frequent adjacent pair; this is the next merge candidate.
pair = max(stats, key=stats.get)
print(pair)

(101, 32)


### Step 4: Define the new token ID
Assign a fresh ID to represent the merged pair in the updated vocabulary.

In [ ]:
# Assign an ID to the new merged token.
i = 0
idx = 128 + i
print(idx)

128


In [17]:
# Decode the selected pair into characters for a human-readable view.
char_pair = (chr(pair[0]), chr(pair[1]))
print(char_pair)

('i', 'l')


### Step 5: Show which pair we are merging
Print the selected pair in both numeric and character form for readability.

In [ ]:
# Show the exact merge operation that will be applied.
print(f"merging {pair} ({char_pair[0]}{char_pair[1]}) into a new token {idx}")

merging (101, 32) (e ) into a new token 128


### Step 6: Perform the merge
Replace all occurrences of the chosen pair with the new token ID in the sequence.

In [ ]:
# Replace every occurrence of the selected pair with the new token ID.
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# Apply one merge step and inspect the updated IDs.
ids = merge(ids, pair, idx)
print(ids)

[84, 104, 128, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 32, 82, 105, 115, 101, 115, 32, 105, 115, 32, 97, 32, 115, 117, 112, 101, 114, 104, 101, 114, 111, 32, 109, 111, 118, 105, 128, 114, 101, 108, 101, 97, 115, 101, 100, 32, 105, 110, 32, 50, 48, 49, 50, 46, 32, 73, 116, 32, 105, 115, 32, 116, 104, 128, 102, 105, 110, 97, 108, 32, 112, 97, 114, 116, 32, 111, 102, 32, 67, 104, 114, 105, 115, 116, 111, 112, 104, 101, 114, 32, 78, 111, 108, 97, 110, 8217, 115, 32, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 32, 116, 114, 105, 108, 111, 103, 121, 44, 32, 102, 111, 108, 108, 111, 119, 105, 110, 103, 32, 66, 97, 116, 109, 97, 110, 32, 66, 101, 103, 105, 110, 115, 32, 97, 110, 100, 32, 84, 104, 128, 68, 97, 114, 107, 32, 75, 110, 105, 103, 104, 116, 46, 32, 84, 104, 128, 102, 105, 108, 109, 32, 115, 116, 97, 114, 115, 32, 67, 104, 114, 105, 115, 116, 105, 97, 110, 32, 66, 97, 108, 128, 97, 115, 32, 66, 114, 117, 99, 128, 87, 97, 121, 110, 101, 47, 66, 97, 116, 109, 97, 110,

### Step 7: Run multiple BPE iterations
Repeat counting, selecting, and merging for a fixed number of iterations to grow the vocabulary.

If we do 20 merges, vocabulary size grows from 128 to 148.

In [ ]:
# Utility: count adjacent pair frequencies.
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

# Utility: apply one merge operation to the token stream.
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# Configure the number of merges based on target vocabulary size.
vocab_size = 148
num_merges = vocab_size - 128
ids = list(tokens)

# Track learned merges as: (token_a, token_b) -> merged_token_id.
merges = {}
for i in range(num_merges):
    # 1) Find the most frequent adjacent pair.
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)

    # 2) Assign a new token ID and log the merge in readable form.
    idx = 128 + i
    char_pair = (chr(pair[0]), chr(pair[1]))
    print(f"merging {pair} ({char_pair[0]}{char_pair[1]}) into a new token {idx}")

    # 3) Apply merge and store the rule.
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (101, 32) (e ) into a new token 128
merging (115, 32) (s ) into a new token 129
merging (97, 110) (an) into a new token 130
merging (105, 110) (in) into a new token 131
merging (116, 104) (th) into a new token 132
merging (100, 32) (d ) into a new token 133
merging (116, 32) (t ) into a new token 134
merging (44, 32) (, ) into a new token 135
merging (101, 114) (er) into a new token 136
merging (115, 116) (st) into a new token 137
merging (101, 110) (en) into a new token 138
merging (97, 114) (ar) into a new token 139
merging (130, 133) () into a new token 140
merging (111, 110) (on) into a new token 141
merging (97, 32) (a ) into a new token 142
merging (114, 101) (re) into a new token 143
merging (46, 32) (. ) into a new token 144
merging (97, 108) (al) into a new token 145
merging (97, 116) (at) into a new token 146
merging (105, 108) (il) into a new token 147


In [ ]:
# Compare lengths before/after BPE merges to estimate compression.
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 2822
ids length: 2197
compression ratio: 1.28X


## Using the tiktoken library
This section compares practical tokenization outputs from GPT-2, GPT-3.5, and GPT-4 encodings.

In [13]:
# Install (if needed) and import tiktoken to compare real tokenizer behavior.
! pip install tiktoken

import tiktoken

# Sample text for tokenizer comparison.
text = "The lion roams in the jungle"

# GPT-2 tokenizer example.
tokenizer_gpt2 = tiktoken.get_encoding("gpt2")

# Encode text to token IDs and decode back for verification.
token_ids_gpt2 = tokenizer_gpt2.encode(text)
decoded_text_gpt2 = tokenizer_gpt2.decode(token_ids_gpt2)

# Decode each ID individually to view token pieces.
tokens_gpt2 = [tokenizer_gpt2.decode([tid]) for tid in token_ids_gpt2]

print("=== GPT-2 Encoding ===")
print("Original Text: ", text)
print("Token IDs:     ", token_ids_gpt2)
print("Tokens:        ", tokens_gpt2)
print("Decoded Text:  ", decoded_text_gpt2)
print()


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


=== GPT-2 Encoding ===
Original Text:  The lion roams in the jungle
Token IDs:      [464, 18744, 686, 4105, 287, 262, 20712]
Tokens:         ['The', ' lion', ' ro', 'ams', ' in', ' the', ' jungle']
Decoded Text:   The lion roams in the jungle



In [ ]:
# GPT-3.5 tokenizer example using model-specific encoding.
tokenizer_gpt35 = tiktoken.encoding_for_model("gpt-3.5-turbo")

token_ids_gpt35 = tokenizer_gpt35.encode(text)
decoded_text_gpt35 = tokenizer_gpt35.decode(token_ids_gpt35)
tokens_gpt35 = [tokenizer_gpt35.decode([tid]) for tid in token_ids_gpt35]

print("=== GPT-3.5 Encoding ===")
print("Original Text: ", text)
print("Token IDs:     ", token_ids_gpt35)
print("Tokens:        ", tokens_gpt35)
print("Decoded Text:  ", decoded_text_gpt35)
print()

=== GPT-3.5 Encoding ===
Original Text:  The lion roams in the jungle
Token IDs:      [791, 40132, 938, 4214, 304, 279, 45520]
Tokens:         ['The', ' lion', ' ro', 'ams', ' in', ' the', ' jungle']
Decoded Text:   The lion roams in the jungle



In [ ]:
# GPT-4 tokenizer example for comparison with GPT-2 and GPT-3.5.
tokenizer_gpt4 = tiktoken.encoding_for_model("gpt-4")

token_ids_gpt4 = tokenizer_gpt4.encode(text)
decoded_text_gpt4 = tokenizer_gpt4.decode(token_ids_gpt4)
tokens_gpt4 = [tokenizer_gpt4.decode([tid]) for tid in token_ids_gpt4]

print("=== GPT-4 Encoding ===")
print("Original Text: ", text)
print("Token IDs:     ", token_ids_gpt4)
print("Tokens:        ", tokens_gpt4)
print("Decoded Text:  ", decoded_text_gpt4)

=== GPT-4 Encoding ===
Original Text:  The lion roams in the jungle
Token IDs:      [791, 40132, 938, 4214, 304, 279, 45520]
Tokens:         ['The', ' lion', ' ro', 'ams', ' in', ' the', ' jungle']
Decoded Text:   The lion roams in the jungle
